# AIxodia — stateless Cloudflare D1 mirror watchdog (Kaggle)

Notebook นี้ทำหน้าที่เดียว: อ่าน SCHEDULE จาก kernel output ของ Notebook หลัก แล้ว mirror `sessions/<id>` และ `config:*` ไป Cloudflare D1 ผ่าน D1 REST ตรง (`POST /client/v4/accounts/{a}/d1/database/{d}/query`). ใช้ token จาก Kaggle Secret `CF_TOKEN` เท่านั้น (REQ-046(3)/REQ-046(6) — ไม่ hardcode, ไม่อ่านจาก cell). ถ้า SCHEDULE หรือ token ไม่พร้อม จะหยุดเงียบและพิมพ์เหตุผล ไม่ยิง request เปล่า.

In [2]:
import os, time, base64, json, urllib.request, urllib.error
from datetime import datetime, timezone

CFG = json.loads(base64.b64decode('ewogICJlbmFibGVkIjogdHJ1ZSwKICAidHVubmVsIjogdHJ1ZSwKICAiY2xvdWRmbGFyZWQiOiAiY2xvdWRmbGFyZWQiLAogICJoZWFydGJlYXRFdmVyeVNlYyI6IDI1LAogICJsb29wU2xlZXBTZWMiOiAxLjAsCiAgImFkb3B0RW52IjogIkNGX1RPS0VOIiwKICAibm9kZXNVcmxUdGxTZWMiOiA2MCwKICAibWF4Qnl0ZXMiOiAxMDAwMDAsCiAgInN0YXRlUHJlZml4RmlsdGVyIjogIiIsCiAgIm1heFR1cm5zUGVyUG9zdCI6IDIwMCwKICAiZDFEYXRhYmFzZSI6ICJhaXhvZGlhIiwKICAic2VuZE9uU3RhcnQiOiB7CiAgICAidHVubmVsX3VybCI6IHRydWUsCiAgICAic2Vzc2lvbnMiOiBmYWxzZQogIH0KfQ==').decode())
CF_TOKEN = os.environ.get(CFG["adoptEnv"], "")
ENTRY = json.loads(base64.b64decode('ewogICJtb2JpbGUiOiB7CiAgICAiZW5hYmxlZCI6IHRydWUsCiAgICAiY2xvdWRmbGFyZV9hcGkiOiAiIiwKICAgICJkMV9kYXRhYmFzZSI6ICJhaXhvZGlhIiwKICAgICJsaXN0ZW4iOiAiMTI3LjAuMC4xOjE4NzkwIiwKICAgICJwdWJsaWNfbGlzdGVuIjogIiIsCiAgICAidHVubmVsIjogdHJ1ZSwKICAgICJjbG91ZGZsYXJlZCI6ICJjbG91ZGZsYXJlZCIsCiAgICAic3luY19jb25maWciOiBmYWxzZSwKICAgICJzeW5jX3Nlc3Npb25zIjogdHJ1ZQogIH0KfQ==').decode())
print("cell 1/6: config ready function=decide_source entry.mobile.listen=%s" % ENTRY["mobile"]["listen"])
print("hint: token มาจาก %s เท่านั้น (ยาว=%d)" % (CFG["adoptEnv"], len(CF_TOKEN)))
# The kernel's own start, recorded in the first code cell so the budget can be
# measured against the whole run. The clone, the Go toolchain and the test suite
# are all spent before the service starts, so counting uptime from the daemon
# would ask for more than Kaggle allows and the kernel would be cut mid-turn.
KERNEL_T0 = time.time()
KERNEL_BUDGET_S = int(os.environ.get("AIX_KERNEL_BUDGET_S", "43200"))  # 12h
HANDOVER_LEAD_S = int(os.environ.get("AIX_HANDOVER_LEAD_S", "1800"))    # 30m before the cap
print("cell 1/6: kernel budget=%ds handover_lead=%ds" % (KERNEL_BUDGET_S, HANDOVER_LEAD_S))



In [3]:
def api_base():
    return "https://api.cloudflare.com/client/v4"

def cf_headers(token):
    return {"Authorization": "Bearer " + token, "Content-Type": "application/json"}

def cf_get(path, token):
    req = urllib.request.Request(api_base() + path, headers=cf_headers(token))
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:
        try:
            return json.loads(e.read().decode())
        except Exception:
            return {"success": False, "errors": [{"code": e.code, "message": "http"}]}

def pick_target(token):
    accs = (cf_get("/accounts", token).get("result") or [])
    if not accs:
        return None, "accounts ว่าง — token ไม่มีสิทธิ์ Account หรือหมดอายุ"
    acc = accs[0]["id"]
    dbs = (cf_get(f"/accounts/{acc}/d1/database", token).get("result") or [])
    want = CFG.get("d1Database", "aixodia")
    db = next((d for d in dbs if d.get("name") == want), dbs[0] if dbs else None)
    if db is None:
        return None, "ไม่พบ D1 ชื่อ %r (มี: %s)" % (want, [d.get("name") for d in dbs])
    return (acc, db["uuid"] if "uuid" in db else db.get("id")), None

def d1_query(acc, dbid, token, sql, params=(), dry=False):
    body = json.dumps({"sql": sql, "params": list(params)}).encode()
    if dry:
        return {"dry": True, "bytes": len(body)}
    req = urllib.request.Request(
        f"{api_base()}/accounts/{acc}/d1/database/{dbid}/query",
        data=body, headers=cf_headers(token))
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            env = json.load(r)
    except urllib.error.HTTPError as e:
        try:
            env = json.loads(e.read().decode())
        except Exception:
            env = {"success": False}
    ok = bool(env.get("success")) and all((s.get("success", True) for s in (env.get("result") or [])))
    if not ok:
        raise RuntimeError("D1 query ล้มเหลว: %r" % (env.get("errors"),))
    return env

def emit(event, **fields):
    rec = {"t": datetime.now(timezone.utc).isoformat(), "event": event}
    rec.update(fields)
    print(json.dumps(rec, ensure_ascii=False))
    return rec

print("cell 2/6: network helpers ready function=cf_get,pick_target,d1_query")

In [4]:
def schedule_text():
    txt = globals().get("SCHEDULE", None)
    if isinstance(txt, str) and txt.strip():
        return txt, "var"
    texts = []
    try:
        ip = get_ipython()
        outs = getattr(ip, "output_hist", {}) or {}
        it = outs.values() if hasattr(outs, "values") else outs
        for cell_out in it:
            try:
                for o in getattr(cell_out, "outputs", []):
                    t = (o.get("text") or "") + (o.get("data", {}).get("text/plain", "") or "")
                    if isinstance(t, str) and t.strip():
                        texts.append(t)
            except Exception:
                continue
    except Exception:
        pass
    blob = chr(10).join(texts)
    return blob, "outputs" if blob.strip() else "missing"

def validate_token(token):
    if not token:
        return "CF_TOKEN ว่าง — ไปที่ Add-ons → Secrets แล้วเพิ่มชื่อนี้ก่อน"
    if len(token.strip()) < 20:
        return "CF_TOKEN สั้นผิดปกติ (%d ตัว) — ตรวจว่า copy มาครบ" % len(token.strip())
    return None

print("cell 3/6: discover helpers ready function=schedule_text,validate_token")

In [5]:
def parse_schedule(text):
    sessions, configs = {}, {}
    current = None
    for raw in (text or "").splitlines():
        line = raw.strip()
        if line.startswith("# SESSION:"):
            current = line.split(":", 1)[1].strip()
            sessions.setdefault(current, [])
            continue
        if line.startswith("# CONFIG:"):
            key = line.split(":", 1)[1].strip()
            configs[key] = configs.get(key, "")
            current = ("config", key)
            continue
        if line.startswith("# END"):
            current = None
            continue
        if not line or current is None:
            continue
        if isinstance(current, tuple) and current[0] == "config":
            configs[current[1]] += raw + chr(10)
        else:
            sessions[current].append(raw)
    out_s = {k: chr(10).join(v).strip() for k, v in sessions.items()}
    out_c = {k: v.strip() for k, v in configs.items()}
    return out_s, out_c

def make_ops(sessions, configs, max_turns=200):
    ops = []
    for sid, body in sessions.items():
        turns = [t for t in [x.strip() for x in body.split(chr(10))] if t][:max_turns]
        ops.append(("sessions_put", sid, turns))
    for key, val in configs.items():
        ops.append(("config_put", "config:" + key, val))
    return ops

print("cell 4/6: schedule parser ready function=parse_schedule,make_ops")

In [6]:
def plan(sessions, configs, nodes_url=None, have_token=True):
    ops = make_ops(sessions, configs, CFG.get("maxTurnsPerPost", 200))
    if not ops:
        return {"action": "skip", "reason": "SCHEDULE ไม่มี SESSION/CONFIG ที่ parse ได้"}
    if not have_token:
        return {"action": "wait", "reason": "รอ CF_TOKEN จาก Secrets", "ops": len(ops)}
    action = "push"
    if nodes_url:
        return {"action": action, "reason": "พร้อมยิง", "ops": len(ops),
                "tunnel_url_seen": nodes_url}
    return {"action": action, "reason": "พร้อมยิง (ยังไม่เห็น nodes URL จะประกาศหลัง push สำเร็จ)",
            "ops": len(ops), "tunnel_url_seen": None}

def push(ops, target, token, limit_bytes=100000):
    acc, dbid = target
    results = []
    for kind, key, val in ops:
        if kind == "config_put":
            if len(val.encode()) > limit_bytes:
                results.append({"key": key, "ok": False, "skipped": "oversize"})
                continue
            d1_query(acc, dbid, token,
                "INSERT INTO state(key,value,updated_at) VALUES(?,?,unixepoch()) "
                "ON CONFLICT(key) DO UPDATE SET value=excluded.value,updated_at=unixepoch()",
                (key, val))
            results.append({"key": key, "ok": True})
        else:
            body = json.dumps({"turns": val}, ensure_ascii=False)
            if len(body.encode()) > limit_bytes:
                results.append({"key": "sessions/" + key, "ok": False, "skipped": "oversize"})
                continue
            d1_query(acc, dbid, token,
                "INSERT INTO state(key,value,updated_at) VALUES(?,?,unixepoch()) "
                "ON CONFLICT(key) DO UPDATE SET value=excluded.value,updated_at=unixepoch()",
                ("sessions/" + key, body))
            results.append({"key": "sessions/" + key, "ok": True})
    return results

def announce(acc, dbid, token, url, version="kaggle-aixodia"):
    d1_query(acc, dbid, token,
        "INSERT INTO nodes(id,tunnel_url,version,heartbeat) VALUES(?,?,?,unixepoch()) "
        "ON CONFLICT(id) DO UPDATE SET tunnel_url=excluded.tunnel_url,version=excluded.version,heartbeat=unixepoch()",
        ("ai", url, version))

def read_nodes_url(acc, dbid, token):
    env = d1_query(acc, dbid, token, "SELECT tunnel_url,heartbeat FROM nodes WHERE id='ai' LIMIT 1")
    try:
        rows = (((env.get("result") or [])[0] or {}).get("results") or [])
        if rows and rows[0].get("tunnel_url"):
            return rows[0]["tunnel_url"], rows[0].get("heartbeat")
    except Exception:
        pass
    return None, None

print("cell 5/6: planner ready function=plan,push,announce,read_nodes_url")

In [7]:
err = validate_token(CF_TOKEN)
text, src = schedule_text()
print("source=%s chars=%d token_len=%d" % (src, len(text), len(CF_TOKEN)))
sessions, configs = parse_schedule(text)

target, target_err = (None, err) if err else pick_target(CF_TOKEN)
nodes_url, nodes_hb = (None, None)
if target and not err:
    try:
        nodes_url, nodes_hb = read_nodes_url(*target, CF_TOKEN)
    except Exception as e:
        emit("d1_nodes_read_failed", error=str(e)[:200])

decision = plan(sessions, configs, nodes_url, have_token=(err is None and target is not None))
print("decision:", json.dumps(decision, ensure_ascii=False))

if decision["action"] == "push":
    pushed = push(make_ops(sessions, configs, CFG.get("maxTurnsPerPost", 200)),
                  target, CF_TOKEN, CFG.get("maxBytes", 100000))
    emit("push_done", ok=sum(1 for r in pushed if r.get("ok")),
         skipped=[r["key"] for r in pushed if not r.get("ok")],
         source=src, nodes_url_seen=nodes_url)
    if nodes_url:
        try:
            announce(*target, CF_TOKEN, nodes_url)
            emit("announce_done", tunnel_url=nodes_url)
        except Exception as e:
            emit("announce_failed", error=str(e)[:200])
elif decision["action"] == "wait":
    emit("waiting", reason=decision["reason"], ops=decision.get("ops", 0))
else:
    emit("skipped", reason=decision["reason"])